# Pemrograman Tugas Akhir
M FARHAN ATHAULLOH - 121450117

SPLIT DATA

Total data (Multiband dan mask)

Rasio Split:

*   Train :70%
*   Test : 15%
*   Valid : 15%



# Connect Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install rasterio

# Split Data

In [ ]:
import os
import shutil
import random

# Path asal data
multiband_src = '/content/drive/MyDrive/Dataset3/Data_Burned_Sentinel-2/MULTIBAND'
mask_src = '/content/drive/MyDrive/Dataset3/Data_Burned_Sentinel-2/MASK'

# Path tujuan base
base_out = '/content/drive/MyDrive/Dataset3/Data_Burned_Sentinel-2/SPLITDATA'

# Buat folder output split
for split in ['TRAIN', 'VAL', 'TEST']:
    os.makedirs(os.path.join(base_out, split, 'MULTIBAND'), exist_ok=True)
    os.makedirs(os.path.join(base_out, split, 'MASK'), exist_ok=True)

# Ambil semua file .tif dari multiband
file_list = [f for f in os.listdir(multiband_src) if f.endswith('.tif')]
file_list.sort()

# Acak file (seed agar reproducible)
random.seed(42)
random.shuffle(file_list)

# Hitung jumlah data
total = len(file_list)
train_count = int(0.7 * total)
val_count = int(0.15 * total)
test_count = total - train_count - val_count
# Split file
train_files = file_list[:train_count]
val_files = file_list[train_count:train_count + val_count]
test_files = file_list[train_count + val_count:]

# Fungsi salin
def copy_files(file_list, split_name):
    for filename in file_list:
        mb_src = os.path.join(multiband_src, filename)
        mask_filename = filename.replace('multiband', 'mask')
        mk_src = os.path.join(mask_src, mask_filename)

        mb_dst = os.path.join(base_out, split_name, 'MULTIBAND', filename)
        mk_dst = os.path.join(base_out, split_name, 'MASK', mask_filename)

        if os.path.exists(mb_src) and os.path.exists(mk_src):
            shutil.copy2(mb_src, mb_dst)
            shutil.copy2(mk_src, mk_dst)
        else:
            print(f"File tidak ditemukan: {filename} atau masknya")

# Salin ke masing-masing folder
copy_files(train_files, 'TRAIN')
copy_files(val_files, 'VAL')
copy_files(test_files, 'TEST')

print(f"Split selesai.")
print(f"Total data: {total}")
print(f"TRAIN: {len(train_files)}, VAL: {len(val_files)}, TEST: {len(test_files)}")

# Menampilkan Distribusi Mask PerPatch

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

# PILIH FOLDER
split = "TRAIN"  # bisa diganti TEST / VAL

# PATH dasar
base_path = "/content/drive/MyDrive/Dataset2/Data_Burned_Sentinel-2/SPLITDATA"
mask_path = os.path.join(base_path, split, "MASK")

# Ambil 1 mask secara acak
sample_mask_name = np.random.choice(os.listdir(mask_path))
sample_mask_path = os.path.join(mask_path, sample_mask_name)

# Baca mask
mask = cv2.imread(sample_mask_path, cv2.IMREAD_UNCHANGED)

# Hitung pixel burned/unburned
unique, counts = np.unique(mask, return_counts=True)

# Siapkan label
label_map = {0: "Unburned (0)", 1: "Burned (1)"}
labels = [label_map.get(u, str(u)) for u in unique]

colors = ["green" if u == 0 else "red" for u in unique]

# Plot
plt.figure(figsize=(8, 6))
bars = plt.bar(labels, counts, color=colors)

plt.title(f"Distribusi Piksel Mask (1 Patch) — {split}", fontsize=14, fontweight="bold")
plt.xlabel("Nilai Piksel Mask", fontsize=12)
plt.ylabel("Frekuensi", fontsize=12)

# Tampilkan angka di atas bar
for bar, val in zip(bars, counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{val:,}".replace(",", "."),
        ha='center', va='bottom', fontsize=11, fontweight="bold"
    )

# Grid horizontal
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

print("Patch dianalisis:", sample_mask_name)
print("Jumlah piksel:", dict(zip(labels, counts)))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
# PILIH FOLDER
split = "TRAIN" # bisa diganti TEST / VAL

# PATH dasar
base_path = "/content/drive/MyDrive/Dataset2/Data_Burned_Sentinel-2/SPLITDATA"
mask_path = os.path.join(base_path, split, "MASK")

# Ambil 1 mask secara acak
sample_mask_name = np.random.choice(os.listdir(mask_path))
sample_mask_path = os.path.join(mask_path, sample_mask_name)

# Baca mask
mask = cv2.imread(sample_mask_path, cv2.IMREAD_UNCHANGED)

# Hitung pixel burned/unburned
unique, counts = np.unique(mask, return_counts=True)

# Siapkan label
label_map = {0: "Unburned (0)", 1: "Burned (1)"}
labels = [label_map.get(u, str(u)) for u in unique]
colors = ["green" if u == 0 else "red" for u in unique]

# AUTO SCALING
max_val = counts.max()
y_limit = max_val * 1.25  # kasih ruang 25%

# PLOTTING
plt.figure(figsize=(6, 5))
bars = plt.bar(labels, counts, color=colors, width=0.5)

plt.title(f"Distribusi Piksel Mask (1 Patch) — {split}", fontsize=14, fontweight="bold")
plt.xlabel("Nilai Piksel Mask", fontsize=11)
plt.ylabel("Frekuensi", fontsize=11)
plt.ylim(0, y_limit)

# Tulis angka di atas bar (auto center)
for bar, val in zip(bars, counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        val + (max_val * 0.02),  # geser sedikit ke atas
        f"{val:,}".replace(",", "."),
        ha='center', fontsize=11, fontweight="bold"
    )

# Grid horizontal
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

print("Patch dianalisis:", sample_mask_name)
print("Jumlah piksel:", dict(zip(labels, counts)))